# Notebook 09 — Lateral Contact Stiffness, Node Branches, and the Physical k₂

## Motivation

Notebook 07 tracks two nulls near the cantilever tip: the on-resonance **displacement
null (D-NS)** and the quasi-static **electrostatic blind spot (ESBS)**. Their behaviour
as contact stiffness increases depends **qualitatively** on the lateral (longitudinal)
contact stiffness k₂:

| model | high-k limit of D-NS | high-k limit of ESBS |
|---|---|---|
| k₂ = 0 (frictionless) | contact − H·tanφ (before the contact) | contact − H·tanφ |
| k₂ = 0.85 k₁ (Mindlin, rigid tip) | saturates ≈3.6 µm **past** the contact (x/L ≈ 0.9525) | approaches the contact point (interior null exists only above k₁ ≈ 10⁵ N/m) |
| k₂ = 0.85k₁ in series with tip shank k_tip | between the two, → contact − H·tanφ | → contact − H·tanφ |

**Which is physical?** Mindlin no-slip theory gives k₂/k₁ = 4G*/E* ≈ 0.85 *at the
contact*, but the beam boundary condition sees this in **series with the lateral bending
stiffness of the tip shank**, k_tip ~ 3EI_tip/H³ ≈ 10²–10⁴ N/m for a tall (15 µm) tip —
and k_tip does not grow with load. So the effective k₂/k₁ collapses at high k₁ and a real
experiment interpolates between the two limits.

All results below use the **two-segment overhang solver** (commit `3aceb6e`): segment 1
clamped→contact with the tip reaction as interior jump conditions, segment 2 a genuinely
free overhang (z″(L) = z‴(L) = 0) that also carries the distributed electrostatic load.
An earlier version of this notebook used the legacy terminal-BC + cubic-extrapolation
overhang treatment, which manufactured a spurious second node, a "fold" (branch
annihilation) in the Mindlin D-NS sweep, and false convergence of the D-NS to the
contact point at high load — Section 2 documents that artifact.

This notebook: (1) sweeps k₁ for the four k₂ models, (2) shows the legacy D-NS "fold"
of the pure Mindlin case was an overhang-treatment artifact and characterises the
corrected smooth branch, and (3) shows how the ESBS-parking protocol errors depend on
the k₂ choice.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../src'))
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from scipy.signal import argrelmin

from eb_cr_afm import (CantileverParams, ContactParams, ExcitationParams, SolverOptions,
                       solve_single_frequency, frequency_sweep)

RESULTS_DIR = os.path.join(os.getcwd(), '../results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- geometry: manuscript figure parameter set -------------------------------
cant = CantileverParams(L=240e-6, w=40e-6, H=15e-6, k=3.0, m=5e-11, mT=5e-12,
                        phi=0.2, tip_offset=15e-6)
XC   = cant.contact_x                 # contact point [m]
XCL  = XC / cant.L                    # ... as x/L
XLIM = (XC - cant.H*np.tan(cant.phi)) / cant.L   # mixed-BC limit: contact - H tan(phi)

opts   = SolverOptions(n_points=600, g=20.0, method='analytical')
exc_pz = ExcitationParams(d1=1e-12, d2=0, f0=0, use_f0=False, bias_coupled_f0=False,
                          delta_V=0.0, V_AC=0.0, eps_r=1.0, use_distributed_load=False)
exc_es = ExcitationParams(d1=0, d2=0, f0=0, use_f0=False, bias_coupled_f0=False,
                          delta_V=1.0, V_AC=1.0, eps_r=1.0, use_distributed_load=True)

# --- k2 models ---------------------------------------------------------------
K_TIPS = (1000.0, 3000.0)   # lateral tip-shank stiffness candidates [N/m]
MODELS = {
    'zero'    : lambda k1: 0.0,
    'mindlin' : lambda k1: 0.85*k1,
    'ktip1000': lambda k1: 1.0/(1.0/(0.85*k1) + 1.0/K_TIPS[0]),
    'ktip3000': lambda k1: 1.0/(1.0/(0.85*k1) + 1.0/K_TIPS[1]),
}
print(f'contact x/L = {XCL:.4f} | contact - H tan(phi) = {XLIM:.4f}')

## Helpers

`find_fcr` locates the contact resonance from the amplitude peak at a reference point
(robust against the ill-conditioning of the analytical BC determinant at high k₁):
a coarse 1 kHz sweep brackets the peak, then a bounded `scipy.optimize.minimize_scalar`
polishes it to ≲0.1 Hz — the resonance linewidth is only a few Hz at g = 20 s⁻¹, so any
fixed frequency grid (the previous version used 100 Hz steps) is too coarse at high k₁.
The frequency window is seeded from the previous k₁ so the same mode branch is tracked.
`minima` returns **all** interior local minima of |z(x)| in the last 70 µm (dense cubic
resampling; no parabolic fit), with normalised depth — so multiple node branches are kept.

In [ ]:
from scipy.optimize import minimize_scalar

DEPTH_THRESH = 0.03   # minima deeper than 3% of window max count as true nulls

def find_fcr(ct, f_lo=80e3, f_hi=700e3):
    """Contact resonance: coarse 1 kHz bracket + bounded scalar refinement."""
    o = SolverOptions(n_points=300, g=20.0, method='analytical',
                      f_start=f_lo, f_stop=f_hi, f_step=1e3)
    sw = frequency_sweep(cant, ct, exc_pz, o, obs_x=cant.L-100e-6)
    f0 = sw['freqs'][int(np.argmax(sw['amplitudes_disp']))]
    o_pt = SolverOptions(n_points=300, g=20.0, method='analytical')
    def neg_amp(f):
        return -solve_single_frequency(f, cant, ct, exc_pz, o_pt,
                                       obs_x=cant.L-100e-6)['A_disp']
    r = minimize_scalar(neg_amp, bounds=(f0-1.2e3, f0+1.2e3),
                        method='bounded', options=dict(xatol=0.05))
    return float(r.x)

def minima(ct, exc, f, win=70e-6):
    r = solve_single_frequency(f, cant, ct, exc, opts)
    x = r['x']; m = x >= cant.L - win
    xs = np.linspace(x[m][0], x[-1], 30001)
    A = np.abs(CubicSpline(x[m], np.real(r['z'][m]))(xs)
               + 1j*CubicSpline(x[m], np.imag(r['z'][m]))(xs))
    return [(xs[i]/cant.L, A[i]/A.max()) for i in argrelmin(A)[0]]

def branch(rows, key):
    """Primary branch by nearest-continuation + secondary nodes."""
    ks, xs, sk, sx = [], [], [], []
    prev = None
    for row in rows:
        mins = [m for m in row[key] if m[1] < DEPTH_THRESH]
        if not mins:
            continue
        x = (min(mins, key=lambda c: abs(c[0]-prev))[0] if prev is not None
             else min(m[0] for m in mins))
        ks.append(row['k1']); xs.append(x); prev = x
        for m in mins:
            if abs(m[0]-x) > 1e-4:
                sk.append(row['k1']); sx.append(m[0])
    return np.array(ks), np.array(xs), np.array(sk), np.array(sx)

## 1 — Sweep k₁ for the four k₂ models

In [ ]:
K1_SWEEP = np.logspace(np.log10(300), np.log10(3e6), 24)
DATA = {}
for name, k2f in MODELS.items():
    rows, f_lo, f_hi = [], 80e3, 560e3
    for k1 in K1_SWEEP:
        ct  = ContactParams(k1=k1, k2=k2f(k1), c1=20e-6, c2=0.0)
        fcr = find_fcr(ct, f_lo, f_hi)
        f_lo, f_hi = max(80e3, fcr-25e3), min(700e3, fcr+30e3)
        rows.append(dict(k1=k1, k2=k2f(k1), fcr=fcr,
                         dns=minima(ct, exc_pz, fcr),
                         es =minima(ct, exc_es, 30e3)))
    DATA[name] = rows
    print(f'{name:9s} done  (f_CR {rows[0]["fcr"]/1e3:.0f} -> {rows[-1]["fcr"]/1e3:.0f} kHz)')

### Fig. S1 — Null positions vs k₁ across k₂ models

- **k₂ = 0.85 k₁** (red): single smooth branch. The D-NS saturates ≈3.6 µm *past* the
  contact point (x/L → 0.9525) and never reaches it, while the ESBS approaches the
  contact point from the overhang side — a persistent D-NS/ESBS separation in the
  hard-indentation limit. (The kink/fold reported here by the legacy solver is gone —
  see Section 2.)
- **k₂ = 0** (blue): smooth, converges to contact − H·tanφ.
- **series models** (orange/purple): follow Mindlin at low k₁, collapse toward the
  frictionless curve at high k₁ — they cross *below* the contact point. This is the
  physically expected behaviour for a tall tip.
- **Missing ESBS points at low k₁**: the electrostatic load on the overhang (the part
  of the lever closest to the sample) removes the interior blind spot altogether at
  soft contact — no minimum deeper than the 3 % threshold exists on the lever below
  k₁ ≈ 1.5 kN/m (k₂ = 0), ≈ 2.2/3.3 kN/m (series models), and ≈ 1.2×10⁵ N/m
  (pure Mindlin: the growing lateral constraint keeps the quasi-static profile
  node-free until far into the hard-contact regime).

In [ ]:
STYLE = [('mindlin',  'k$_2$ = 0.85 k$_1$ (Mindlin, rigid tip)',       '#c00000'),
         ('ktip3000', 'k$_2$ = 0.85k$_1$ $\\parallel$ k$_{tip}$=3 kN/m', '#e67e22'),
         ('ktip1000', 'k$_2$ = 0.85k$_1$ $\\parallel$ k$_{tip}$=1 kN/m', '#7030a0'),
         ('zero',     'k$_2$ = 0 (frictionless)',                     '#1f6cb5')]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, key, ttl in zip(axes, ('dns', 'es'),
                        ('D-NS (on-resonance displacement null)',
                         'ESBS (30 kHz electrostatic blind spot)')):
    for name, lbl, col in STYLE:
        ks, xs, sk, sx = branch(DATA[name], key)
        ax.semilogx(ks, xs, '-', color=col, lw=2.2, label=lbl)
        if len(sk):
            ax.plot(sk, sx, 's', color=col, ms=4, mfc='none')
    ax.axhline(XCL,  color='green', ls='--', lw=1.2)
    ax.axhline(XLIM, color='green', ls=':',  lw=1.2)
    ax.set_xlabel('Contact stiffness k$_1$ (N/m)'); ax.set_title(ttl, fontsize=11)
    ax.grid(alpha=0.25, which='both')
axes[0].set_ylabel('Null position x/L')
axes[0].text(320, XCL+0.0015, 'contact point', color='green', fontsize=9)
axes[0].text(320, XLIM+0.0015, r'contact $-$ H tan$\varphi$', color='green', fontsize=9)
axes[0].legend(fontsize=9, loc='upper right'); axes[0].set_ylim(0.920, 1.002)
fig.suptitle('Fig. S1 — Null positions vs contact stiffness for four lateral-stiffness models', fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(os.path.join(RESULTS_DIR, 'nb09_figS1_k2_models.png'), dpi=180, bbox_inches='tight')
plt.show()

## 2 — The legacy D-NS "fold" was an overhang-treatment artifact

Earlier versions of this notebook (and of the solver) applied Bradler's *terminal*
boundary conditions at the contact point and reconstructed the overhang by a cubic
Taylor extrapolation of the contact-point derivatives. That assigns the **tip-reaction
moment and shear to the load-free overhang**, which can physically carry only its own
inertial moment — a fraction ~|αL₂|⁴ ≈ 10⁻² of the tip reaction. The extrapolated field
violates the free-end conditions z″(L) = z‴(L) = 0 by orders of magnitude and
manufactures spurious displacement nodes exactly on the stretch of beam where the
near-tip nulls live. For k₂ = 0.85 k₁ this produced: a false second node entering
through the free end at k₁ ≈ 6 kN/m, a false node annihilation ("fold") at k₁ ≈ 8 kN/m,
and false convergence of the D-NS to the contact point at high load. (The `solve_bvp`
cross-check that previously appeared to confirm the fold applied the same terminal-BC
+ extrapolation treatment, so it reproduced the same artifact.)

The corrected **two-segment model** — free overhang with interior jump conditions,
independently verified against a Hermite-FEM discretization and a genuinely two-segment
`solve_bvp` (see `docs/model.md`) — shows instead:

1. a **single node** in the on-resonance tip profiles that migrates smoothly with k₁
   through the former "fold" region — no node entry, no annihilation (panel a);
2. at the specific stiffness where the legacy treatment produced its spurious second
   node, the two-segment and Hermite-FEM profiles agree and show one node (panel b);
3. a **smooth D-NS branch** that saturates at x/L ≈ 0.9525 (≈3.6 µm past the contact)
   instead of folding and dropping to the contact point (panel c).

In [ ]:
KS_FOLD = [5423., 6186., 7056., 8048., 9180.]   # former fold region
import matplotlib.cm as cm
cols = cm.viridis(np.linspace(0.1, 0.85, len(KS_FOLD)))
# Archived legacy-vs-corrected-vs-FEM profiles (shipped with the repository)
PAPER_NPZ = os.path.join('data', 'legacy_overhang_comparison.npz')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

# (a) corrected on-resonance near-tip profiles through the former fold region
ax = axes[0]
for k1, ccol in zip(KS_FOLD, cols):
    ct  = ContactParams(k1=k1, k2=0.85*k1, c1=20e-6, c2=0.0)
    fcr = find_fcr(ct, 450e3, 600e3)
    r   = solve_single_frequency(fcr, cant, ct, exc_pz, opts)
    x   = r['x']/cant.L; m = x >= 0.93
    xs  = np.linspace(x[m][0], 1.0, 8001)
    A   = np.abs(CubicSpline(x[m], np.real(r['z'][m]))(xs)
                 + 1j*CubicSpline(x[m], np.imag(r['z'][m]))(xs))
    ax.plot(xs, A/np.max(np.abs(r['z'])), color=ccol, lw=1.8,
            label=f'{k1/1e3:.1f} kN/m')
ax.axvline(XCL, color='green', ls='--', lw=1.0)
ax.set_xlabel('x/L'); ax.set_ylabel('|z(x)| / max (on-res)'); ax.set_ylim(0, 0.06)
ax.set_title('(a) Corrected profiles through former fold\nregion: one node, smooth migration', fontsize=10)
ax.legend(fontsize=8, title='k$_1$')

# (b) legacy vs corrected vs FEM profile at k1 = 7.1 kN/m (archived, figM1a)
ax = axes[1]
if os.path.exists(PAPER_NPZ):
    dM = np.load(PAPER_NPZ)
    sc = np.max(dM['profile_A_corr'])
    ax.plot(dM['profile_x_leg']/cant.L,  dM['profile_A_leg']/sc,  '--', color='0.45',
            lw=1.8, label='legacy: terminal BC + cubic overhang')
    ax.plot(dM['profile_x_corr']/cant.L, dM['profile_A_corr']/sc, '-',  color='#c00000',
            lw=2.0, label='two-segment (corrected)')
    ax.plot(dM['profile_x_fem']/cant.L,  dM['profile_A_fem']/sc,  'o',  color='k',
            ms=4, mfc='none', label='Hermite FEM')
    ax.axvline(XCL, color='green', ls='--', lw=1.0)
    ax.set_xlabel('x/L'); ax.set_ylabel('|z(x)| (norm.)')
    ax.set_title('(b) k$_1$ = 7.1 kN/m: legacy manufactures\na spurious second node', fontsize=10)
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'data/legacy_overhang_comparison.npz not found', ha='center', va='center',
            transform=ax.transAxes)

# (c) D-NS branch: corrected (Section 1 sweep) vs legacy (archived)
ax = axes[2]
ks, xs_b, _, _ = branch(DATA['mindlin'], 'dns')
ax.semilogx(ks, xs_b, '-o', color='#c00000', ms=4, lw=2,
            label='two-segment (corrected)')
if os.path.exists(PAPER_NPZ):
    dM = np.load(PAPER_NPZ)
    ax.semilogx(dM['leg_k'], dM['leg_x'], 's--', color='0.45', ms=4, lw=1.4,
                mfc='none', label='legacy (fold artifact)')
    if len(dM['leg_sk']):
        ax.plot(dM['leg_sk'], dM['leg_sx'], 's', color='0.45', ms=4, mfc='none')
ax.axhline(XCL,  color='green', ls='--', lw=1.0)
ax.axhline(XLIM, color='green', ls=':',  lw=1.0)
ax.text(3.5e2, XCL+0.0015, 'contact point', color='green', fontsize=8)
ax.text(3.5e2, XLIM+0.0015, r'contact $-$ H tan$\varphi$', color='green', fontsize=8)
ax.set_xlabel('k$_1$ (N/m)'); ax.set_ylabel('D-NS position x/L')
ax.set_title('(c) D-NS branch: smooth saturation vs\nlegacy fold + false convergence', fontsize=10)
ax.legend(fontsize=8, loc='lower right')

fig.suptitle('Fig. S2 — The legacy D-NS "fold" (k$_2$ = 0.85 k$_1$) is an artifact of the overhang treatment', fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(os.path.join(RESULTS_DIR, 'nb09_figS2_fold_anatomy.png'), dpi=180, bbox_inches='tight')
plt.show()

## 3 — ESBS-parking protocol errors across the k₂ models

Protocol: find the ESBS at high k₁ (30 kN/m), park the detector there, measure
sub-resonance (30 kHz) at lower k₁. Error channels: (i) residual **electrostatic
leakage** at the parked spot (the true ESBS migrates with k₁), per V_AC·ΔV = 1 V²
against a d₁ = 1 pm piezo signal; (ii) **geometric error** of the mechanical signal
(parked spot vs true contact-point displacement).

**Caveat for the pure Mindlin model**: with the corrected solver there is *no interior
ESBS at k₁ = 30 kN/m* (it only forms above k₁ ≈ 1.2×10⁵ N/m), so `null1` parks at the
profile minimum, which sits at the free end x = L. The "ESBS moved" column is then 0 by
construction (the 30 kHz minimum stays at the free end for all k₁ ≤ 30 kN/m) and the
geometric error reflects the offset between the free end and the contact point.

In [ ]:
def zat(r, xq):
    return (CubicSpline(r['x'], np.real(r['z']))(xq)
            + 1j*CubicSpline(r['x'], np.imag(r['z']))(xq))

def null1(r, win=70e-6):
    x = r['x']; m = x >= cant.L - win
    xs = np.linspace(x[m][0], x[-1], 40001)
    A = np.abs(CubicSpline(x[m], np.real(r['z'][m]))(xs)
               + 1j*CubicSpline(x[m], np.imag(r['z'][m]))(xs))
    return xs[int(np.argmin(A))]

print(f'{"model":>9} | {"k1":>6} {"ESBS moved":>10} {"ES leak pm/V2":>13} {"contam %":>9} {"geom %":>7}')
for name in ('zero', 'ktip3000', 'mindlin'):
    k2f = MODELS[name]
    ct_hi  = ContactParams(k1=30000., k2=k2f(30000.), c1=20e-6, c2=0.0)
    x_park = null1(solve_single_frequency(30e3, cant, ct_hi, exc_es, opts))
    for k1 in (300., 1000., 3000., 10000., 30000.):
        ct   = ContactParams(k1=k1, k2=k2f(k1), c1=20e-6, c2=0.0)
        r_es = solve_single_frequency(30e3, cant, ct, exc_es, opts)
        r_pz = solve_single_frequency(30e3, cant, ct, exc_pz, opts)
        xb   = null1(r_es)
        zes, zpz, zc = zat(r_es, x_park), zat(r_pz, x_park), zat(r_pz, XC)
        print(f'{name:>9} | {k1:6.0f} {abs(xb-x_park)*1e6:9.2f}u {abs(zes)*1e12:13.3f} '
              f'{abs(zes)/abs(zpz)*100:9.1f} {abs(zpz-zc)/abs(zc)*100:7.2f}')
    print('-'*62)

## Conclusions

1. **The k₂ model matters qualitatively** for where the nulls converge at high load.
   With frictionless contact (k₂ = 0) both nulls converge to contact − H·tanφ
   (x/L ≈ 0.925). With rigid Mindlin coupling (k₂ = 0.85 k₁) the D-NS saturates
   ≈3.6 µm **past** the contact (x/L ≈ 0.9525) and never reaches it, while the ESBS
   approaches the contact point — a persistent D-NS/ESBS separation in the
   hard-indentation limit. With the physically expected series combination (Mindlin
   contact + tip-shank lateral compliance, k_tip ~ 1–3 kN/m for a 15 µm tip), the
   effective k₂/k₁ collapses at high load and both nulls cross below the contact
   point toward contact − H·tanφ. **The k₂ = 0 figure remains the better high-k
   approximation for a tall tip**; its tilt-bias conclusion (≈ H·tanφ ≈ 2–3 µm below
   contact at 30 kN/m) survives realistic shank compliance.

2. **The D-NS "fold" of the pure Mindlin sweep, reported by earlier versions of this
   notebook, was a solver artifact** of the legacy terminal-BC + cubic-overhang
   treatment (Section 2). The corrected two-segment model gives a single node and a
   smooth branch: no second node, no annihilation, and no convergence to the contact
   point (the legacy branch fell to the contact point at high k₁; the corrected one
   saturates at x/L ≈ 0.9525). The `solve_bvp` "verification" quoted previously
   shared the same flawed overhang treatment; the corrected solver is instead
   verified against a Hermite-FEM discretization and a genuinely two-segment
   `solve_bvp` (see `docs/model.md`).

3. **ESBS-parking protocol errors.** For the frictionless and series models the
   picture is unchanged in kind: electrostatic leakage at the parked spot dominates
   at low load (~30 % at 300 N/m, ~3 % at 3 kN/m, <1 % above 10 kN/m per
   V_AC·ΔV = 1 V², scaling linearly with V_AC·ΔV), while the geometric
   (tilt-projection) error stays ≤1.6 % at all k₁. Two corrected-solver caveats:
   (i) because the electrostatic load acts on the overhang, the interior ESBS
   disappears entirely at soft contact (below ≈1.5–3.3 kN/m depending on k₂), so the
   parked spot cannot be *defined* from a low-load calibration; (ii) for pure rigid
   Mindlin coupling there is no interior ESBS even at 30 kN/m — the protocol then
   parks at the free end and the mechanical signal there misrepresents the
   contact-point displacement by 4–19 %. The parking protocol is therefore
   well-defined only in a k₂-dependent stiffness window, and the practical
   prescription remains: calibrate the blind spot in the hard-indentation regime.